# Chapter 4: Transformer & Attention Mathematics

**Mathematics Behind LLMs — Book Series**

This chapter derives the mathematics inside the transformer architecture from first principles: how tokens are embedded and positioned, how attention computes context-aware representations, and how the full forward pass flows through normalization, feed-forward layers, and residual connections.

---

**Topics covered:**
1. Token Embeddings & Sinusoidal Positional Encoding
2. Scaled Dot-Product Attention
3. Multi-Head Attention
4. Rotary Position Encoding (RoPE)
5. LayerNorm & RMSNorm
6. Feed-Forward Networks with SwiGLU
7. Full Transformer Block
8. Parameter Count Verification
9. KV Cache Demo

In [ ]:
# Setup: imports and reproducibility
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print("Seed set to 42 for reproducibility.")

## 4.1 Token Embeddings & Sinusoidal Positional Encoding

**Token embeddings** map discrete token IDs to continuous vectors. The embedding matrix $\mathbf{E} \in \mathbb{R}^{|V| \times d}$ is learned; for token $id_t$:

$$\mathbf{x}_t = \mathbf{E}[id_t] \in \mathbb{R}^d$$

**The problem:** Transformers are permutation-equivariant — without positional information, the model treats the sequence $[A, B, C]$ identically to $[C, A, B]$. We must inject position information.

**Sinusoidal Positional Encoding** (Vaswani et al., 2017) adds fixed, non-learned vectors:

$$\text{PE}(pos, 2i) = \sin\!\left(\frac{pos}{10000^{2i/d}}\right)$$

$$\text{PE}(pos, 2i+1) = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

where $pos$ is the token position and $i \in \{0, 1, \ldots, d/2-1\}$ indexes the embedding dimension.

**Why sinusoids?**
- Different frequencies encode different scales of position (low freq = coarse, high freq = fine)
- $\text{PE}(pos+k)$ can be expressed as a linear function of $\text{PE}(pos)$ — allows the model to easily attend to relative positions
- Generalizes to sequence lengths longer than seen during training

The final input embedding is: $\mathbf{h}_t^{(0)} = \mathbf{x}_t + \text{PE}(t)$

In [ ]:
# Token embeddings + sinusoidal positional encoding

torch.manual_seed(42)

def sinusoidal_pe(max_seq_len, d_model):
    """
    Compute sinusoidal positional encoding matrix.
    Returns: tensor of shape (max_seq_len, d_model)
    """
    pe = torch.zeros(max_seq_len, d_model)
    positions = torch.arange(0, max_seq_len, dtype=torch.float32).unsqueeze(1)  # (T, 1)
    # Frequencies: 1 / 10000^(2i/d)
    dim_indices = torch.arange(0, d_model, 2, dtype=torch.float32)  # even indices: 0, 2, 4, ...
    div_term = torch.exp(-dim_indices * math.log(10000.0) / d_model)  # (d/2,)
    pe[:, 0::2] = torch.sin(positions * div_term)  # even dims: sin
    pe[:, 1::2] = torch.cos(positions * div_term)  # odd dims:  cos
    return pe  # (T, d)

# Hyperparameters
B = 2       # batch size
T = 8       # sequence length
d = 16      # model dimension (d_model)
V = 100     # vocabulary size

# Token embedding lookup
embedding = nn.Embedding(V, d)
token_ids = torch.randint(0, V, (B, T))  # (B, T)
token_emb = embedding(token_ids)          # (B, T, d)
print(f"Token IDs shape:       {token_ids.shape}")
print(f"Token embeddings shape: {token_emb.shape}")

# Sinusoidal PE
pe = sinusoidal_pe(T, d)  # (T, d)
print(f"\nPositional encoding shape: {pe.shape}")
print(f"PE range: [{pe.min():.3f}, {pe.max():.3f}]  (should be in [-1, 1])")

# Add PE to token embeddings (broadcast over batch)
h = token_emb + pe.unsqueeze(0)  # (B, T, d)
print(f"\nInput embeddings h = token_emb + PE: {h.shape}")

# Show sinusoidal pattern
print("\nFirst 4 positions, first 8 dimensions of PE:")
print("  Pos | " + "  ".join([f"dim{i}" for i in range(8)]))
for pos in range(4):
    vals = pe[pos, :8].round(decimals=3).tolist()
    print(f"  {pos:3d} | {vals}")

# Verify periodicity: sin and cos alternate
pe_full = sinusoidal_pe(64, 32)
print(f"\nFull PE shape for (64 positions, 32 dims): {pe_full.shape}")
print(f"All values in [-1, 1]: {pe_full.abs().max().item() <= 1.0 + 1e-6}")

## 4.2 Scaled Dot-Product Attention

**Attention** allows each token to selectively aggregate information from all other tokens. Given queries $\mathbf{Q}$, keys $\mathbf{K}$, and values $\mathbf{V}$:

$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}}\right)\mathbf{V}$$

where $\mathbf{Q}, \mathbf{K} \in \mathbb{R}^{T \times d_k}$ and $\mathbf{V} \in \mathbb{R}^{T \times d_v}$.

**Why scale by $\sqrt{d_k}$?**

If $\mathbf{q}$ and $\mathbf{k}$ are random vectors with unit variance, their dot product $\mathbf{q} \cdot \mathbf{k} = \sum_{i=1}^{d_k} q_i k_i$ has variance $d_k$ (sum of $d_k$ unit-variance terms). Without scaling, for large $d_k$ (e.g., $d_k = 64$), the dot products push the softmax into saturation regions where gradients vanish. Dividing by $\sqrt{d_k}$ restores variance to $O(1)$.

**Causal (decoder) mask:** In autoregressive LMs, token $t$ must not attend to future tokens $t' > t$. We add $-\infty$ to the upper triangle of the attention matrix before softmax:

$$A_{ij} = \begin{cases} \frac{\mathbf{q}_i \cdot \mathbf{k}_j}{\sqrt{d_k}} & j \leq i \\ -\infty & j > i \end{cases}$$

After softmax, positions with $-\infty$ logits get zero weight — causal masking enforced.

In [ ]:
# Scaled dot-product attention with causal masking

torch.manual_seed(42)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled dot-product attention.
    
    Args:
        Q: (B, T_q, d_k)  -- queries
        K: (B, T_k, d_k)  -- keys
        V: (B, T_k, d_v)  -- values
        mask: (T_q, T_k) boolean mask; True = mask out (set to -inf)
    Returns:
        output: (B, T_q, d_v)
        attn_weights: (B, T_q, T_k)
    """
    d_k = Q.shape[-1]
    # Compute attention scores: (B, T_q, T_k)
    scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(d_k)
    # Apply mask (e.g., causal mask)
    if mask is not None:
        scores = scores.masked_fill(mask.unsqueeze(0), float('-inf'))
    # Softmax over key dimension
    attn_weights = F.softmax(scores, dim=-1)
    # Weighted sum of values: (B, T_q, d_v)
    output = torch.bmm(attn_weights, V)
    return output, attn_weights

B, T, d_k, d_v = 2, 6, 8, 8
Q = torch.randn(B, T, d_k)
K = torch.randn(B, T, d_k)
V = torch.randn(B, T, d_v)

# --- Non-causal (bidirectional) attention ---
out_bi, attn_bi = scaled_dot_product_attention(Q, K, V, mask=None)
print(f"Bidirectional attention output: {out_bi.shape}")
print(f"Attention weights shape:        {attn_bi.shape}")
print(f"Attn weights sum to 1 per row:  {attn_bi[0].sum(dim=-1).round(decimals=4).tolist()}")

# --- Causal (autoregressive) mask ---
causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)  # upper triangle = True
print(f"\nCausal mask (True = masked out):")
print(causal_mask.int())

out_causal, attn_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
print(f"\nCausal attention output: {out_causal.shape}")
print("Causal attention weights (batch 0):")
for row in attn_causal[0]:
    print(" ", row.round(decimals=3).tolist())

# Verify causality: weights in upper triangle should be 0
upper_tri_weights = attn_causal[:, causal_mask]
print(f"\nMax weight in masked positions: {upper_tri_weights.max().item():.2e}  (should be ~0)")

# Variance before/after scaling
Q_test = torch.randn(100, d_k)
K_test = torch.randn(100, d_k)
raw_scores = (Q_test @ K_test.T)  # (100, 100)
scaled_scores = raw_scores / math.sqrt(d_k)
print(f"\nDot product variance (d_k={d_k}): raw={raw_scores.var():.3f}, scaled={scaled_scores.var():.3f}")
print(f"Expected: raw~{d_k}, scaled~1")

## 4.3 Multi-Head Attention

**Multi-head attention** runs $h$ attention heads in parallel, each operating on a different learned projection of Q, K, V:

$$\text{MHA}(\mathbf{X}) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\, \mathbf{W}^O$$

where each head is:

$$\text{head}_i = \text{Attention}(\mathbf{X}\mathbf{W}_i^Q,\; \mathbf{X}\mathbf{W}_i^K,\; \mathbf{X}\mathbf{W}_i^V)$$

with $\mathbf{W}_i^Q, \mathbf{W}_i^K, \mathbf{W}_i^V \in \mathbb{R}^{d \times d_k}$ and $d_k = d/h$.

**Parameter count:**
- $\mathbf{W}^Q$: $d \times d$ (combining all heads) $= d^2$
- $\mathbf{W}^K$: $d^2$
- $\mathbf{W}^V$: $d^2$
- $\mathbf{W}^O$: $d^2$
- **Total per MHA layer: $4d^2$** (no bias)

**Why multiple heads?**
- Different heads can attend to different types of relationships (syntactic, semantic, positional)
- Head 1 might specialize in subject-verb agreement; head 2 in coreference; etc.
- Empirically critical: removing heads hurts performance

In [ ]:
# Multi-Head Attention from scratch

torch.manual_seed(42)

class MultiHeadAttention(nn.Module):
    """Multi-Head Attention (MHA) from scratch."""
    
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # per-head dimension
        
        # Projection matrices (no bias for simplicity, matching many LLMs)
        self.W_q = nn.Linear(d_model, d_model, bias=False)  # d x d
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x, mask=None):
        """
        x: (B, T, d_model)
        Returns: (B, T, d_model)
        """
        B, T, d = x.shape
        h = self.num_heads
        d_k = self.d_k
        
        # Project and split into heads
        Q = self.W_q(x).view(B, T, h, d_k).transpose(1, 2)  # (B, h, T, d_k)
        K = self.W_k(x).view(B, T, h, d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, h, d_k).transpose(1, 2)
        
        # Scaled dot-product attention per head
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (B, h, T, T)
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        attn = F.softmax(scores, dim=-1)  # (B, h, T, T)
        
        # Weighted sum: (B, h, T, d_k)
        context = torch.matmul(attn, V)
        
        # Concatenate heads and project
        context = context.transpose(1, 2).contiguous().view(B, T, d)  # (B, T, d)
        output = self.W_o(context)
        return output, attn

# Test
B, T, d_model, num_heads = 2, 10, 64, 8
mha = MultiHeadAttention(d_model, num_heads)

x = torch.randn(B, T, d_model)
causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)

output, attn_weights = mha(x, mask=causal_mask)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}  (should be (B={B}, T={T}, d={d_model}))")
print(f"Attn weights: {attn_weights.shape}  (B, heads, T, T)")

# Parameter count
total_params = sum(p.numel() for p in mha.parameters())
expected_params = 4 * d_model * d_model  # 4d^2
print(f"\nActual parameters:   {total_params:,}")
print(f"Expected (4d^2):     {expected_params:,}  (d={d_model})")
print(f"Match: {total_params == expected_params}")

# Breakdown
for name, p in mha.named_parameters():
    print(f"  {name}: {p.shape} = {p.numel():,} params")

## 4.4 Rotary Position Encoding (RoPE)

**RoPE** (Su et al., 2021) encodes absolute positions in the query/key vectors via rotation, but with the crucial property that the **dot product $\mathbf{q}_m^T \mathbf{k}_n$ depends only on the relative position $m - n$**.

For a 2D vector $(x, y)$ at position $m$, RoPE applies a rotation by angle $m\theta$:

$$\begin{pmatrix} x' \\ y' \end{pmatrix} = \begin{pmatrix} \cos m\theta & -\sin m\theta \\ \sin m\theta & \cos m\theta \end{pmatrix} \begin{pmatrix} x \\ y \end{pmatrix}$$

For $d$-dimensional vectors, RoPE pairs consecutive dimensions $(2i, 2i+1)$ and rotates each pair at frequency $\theta_i = 10000^{-2i/d}$:

$$\mathbf{q}_m^{(i)} = q_{2i} e^{im\theta_i} + q_{2i+1}\, ie^{im\theta_i}$$

**Implementation via complex numbers:** Treat the pair $(q_{2i}, q_{2i+1})$ as a complex number $q_{2i} + iq_{2i+1}$, then multiply by $e^{im\theta_i} = \cos(m\theta_i) + i\sin(m\theta_i)$.

**Why RoPE is widely adopted (LLaMA, PaLM, Gemini, Mistral):**
- Encodes relative position naturally through the dot-product
- Generalizes well to longer contexts via NTK-aware scaling
- No learned parameters (pure geometric transformation)
- Compatible with attention caching

In [ ]:
# Rotary Position Encoding (RoPE) implementation

torch.manual_seed(42)

def precompute_freqs_cis(dim, max_seq_len, base=10000.0):
    """
    Precompute complex exponentials for RoPE.
    Returns: freqs_cis of shape (max_seq_len, dim//2) complex
    """
    # Frequencies: theta_i = 1 / 10000^(2i/d)  for i in [0, d/2)
    freqs = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))  # (dim//2,)
    # Positions: 0, 1, ..., max_seq_len-1
    t = torch.arange(max_seq_len, dtype=torch.float32)  # (T,)
    # Outer product: (T, dim//2)
    freqs_outer = torch.outer(t, freqs)  # m * theta_i
    # Complex exponential: e^{i*m*theta} = cos(m*theta) + i*sin(m*theta)
    freqs_cis = torch.polar(torch.ones_like(freqs_outer), freqs_outer)  # (T, dim//2)
    return freqs_cis

def apply_rotary_emb(q, k, freqs_cis):
    """
    Apply RoPE to queries and keys.
    q, k: (B, T, d_k)  -- real-valued
    freqs_cis: (T, d_k//2)  -- complex
    Returns: q_rot, k_rot of shape (B, T, d_k)
    """
    # Reshape to complex: (B, T, d_k//2) complex
    q_complex = torch.view_as_complex(q.float().reshape(*q.shape[:-1], -1, 2))
    k_complex = torch.view_as_complex(k.float().reshape(*k.shape[:-1], -1, 2))
    
    # Broadcast freqs_cis: (1, T, d_k//2)
    freqs = freqs_cis.unsqueeze(0)  # (1, T, d_k//2)
    
    # Apply rotation: multiply complex vectors
    q_rot = torch.view_as_real(q_complex * freqs).flatten(-2)  # (B, T, d_k)
    k_rot = torch.view_as_real(k_complex * freqs).flatten(-2)  # (B, T, d_k)
    
    return q_rot.type_as(q), k_rot.type_as(k)

# Test RoPE
B, T, d_k = 2, 8, 16
freqs_cis = precompute_freqs_cis(d_k, T)
print(f"freqs_cis shape: {freqs_cis.shape}  (T={T}, d_k//2={d_k//2}) complex")

q = torch.randn(B, T, d_k)
k = torch.randn(B, T, d_k)

q_rot, k_rot = apply_rotary_emb(q, k, freqs_cis)
print(f"q shape:     {q.shape}")
print(f"q_rot shape: {q_rot.shape}  (same as input)")

# Verify: rotation preserves vector norms
q_norms = q.norm(dim=-1)
q_rot_norms = q_rot.norm(dim=-1)
print(f"\nNorm preservation check:")
print(f"  Max norm difference: {(q_norms - q_rot_norms).abs().max().item():.2e}  (should be ~0)")

# Verify relative position property: q_m . k_n depends on (m-n)
# Test: dot product of q at pos=5 with k at pos=2 vs q at pos=3 with k at pos=0
# Both have relative distance 3, so should be similar after rotation
dot_53 = (q_rot[0, 5] * k_rot[0, 2]).sum().item()  # positions 5, 2 -> relative=3
dot_30 = (q_rot[0, 3] * k_rot[0, 0]).sum().item()  # positions 3, 0 -> relative=3
# (These won't be exactly equal since q/k values differ, but position encoding is consistent)
print(f"\nDot products at relative distance 3:")
print(f"  q[5] . k[2] = {dot_53:.4f}")
print(f"  q[3] . k[0] = {dot_30:.4f}")
print("(Different because q/k content differs, but rotation structure is consistent)")

# Show frequencies used
print(f"\nRoPE frequencies (first 5): {freqs_cis[0, :5].abs().round(decimals=4).tolist()}")
print(f"All frequencies have magnitude 1: {torch.allclose(freqs_cis.abs(), torch.ones_like(freqs_cis.abs()))}")

## 4.5 LayerNorm & RMSNorm

Normalization layers stabilize training by controlling the scale of activations.

**LayerNorm** (Ba et al., 2016) normalizes each token's feature vector across the $d$ dimensions:

$$\hat{x}_i = \gamma \cdot \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

where $\mu = \frac{1}{d}\sum_j x_j$ and $\sigma^2 = \frac{1}{d}\sum_j (x_j - \mu)^2$ are computed per token, and $\gamma, \beta \in \mathbb{R}^d$ are learned affine parameters.

**RMSNorm** (Zhang & Sennrich, 2019) simplifies LayerNorm by removing the mean centering:

$$\hat{x}_i = \frac{x_i}{\sqrt{\frac{1}{d}\|\mathbf{x}\|^2 + \epsilon}} \cdot \gamma_i$$

RMS = Root Mean Square: $\text{RMS}(\mathbf{x}) = \sqrt{\frac{1}{d}\sum_i x_i^2}$

**Why RMSNorm?** (Used in LLaMA, PaLM, T5)
- ~$1.3\times$ faster (no mean computation)
- Empirically matches or exceeds LayerNorm quality
- Simpler gradient computation

**Pre-norm vs post-norm:** Modern LLMs use **pre-norm** — applying normalization *before* attention/FFN rather than after. Pre-norm has better gradient flow and training stability.

In [ ]:
# LayerNorm and RMSNorm from scratch

torch.manual_seed(42)

class LayerNormScratch(nn.Module):
    """LayerNorm from scratch."""
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))   # scale
        self.beta = nn.Parameter(torch.zeros(d_model))   # shift
        self.eps = eps
    
    def forward(self, x):
        # x: (B, T, d_model)
        mean = x.mean(dim=-1, keepdim=True)       # (B, T, 1)
        var = x.var(dim=-1, keepdim=True, unbiased=False)  # (B, T, 1)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta

class RMSNorm(nn.Module):
    """RMSNorm from scratch (used in LLaMA, T5, PaLM)."""
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.eps = eps
    
    def forward(self, x):
        # x: (B, T, d_model)
        # RMS(x) = sqrt(mean(x^2))
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)  # (B, T, 1)
        return self.gamma * (x / rms)

# Test on random activations
B, T, d_model = 2, 8, 32
x = torch.randn(B, T, d_model) * 5 + 2  # mean != 0, std != 1

ln_scratch = LayerNormScratch(d_model)
ln_torch = nn.LayerNorm(d_model)
rms_norm = RMSNorm(d_model)

# Copy weights so they're identical
ln_torch.weight.data = ln_scratch.gamma.data.clone()
ln_torch.bias.data = ln_scratch.beta.data.clone()

out_scratch = ln_scratch(x)
out_torch = ln_torch(x)
out_rms = rms_norm(x)

print(f"Input stats: mean={x.mean():.3f}, std={x.std():.3f}")
print(f"LayerNorm output stats: mean={out_scratch.mean():.3e}, std={out_scratch.std():.3f}")
print(f"RMSNorm output stats:   mean={out_rms.mean():.3f}, std={out_rms.std():.3f}")

# Verify scratch matches PyTorch
print(f"\nMax diff (scratch vs PyTorch LayerNorm): {(out_scratch - out_torch).abs().max().item():.2e}")
print(f"Match: {torch.allclose(out_scratch, out_torch, atol=1e-5)}")

# Parameter count
print(f"\nLayerNorm params: {sum(p.numel() for p in ln_scratch.parameters())} = 2*d = 2*{d_model}")
print(f"RMSNorm params:   {sum(p.numel() for p in rms_norm.parameters())} = d = {d_model}")

# Gradient flow demo: RMSNorm provides stable gradient magnitudes
x_test = torch.randn(B, T, d_model, requires_grad=True)
out_rms_test = rms_norm(x_test)
loss = out_rms_test.sum()
loss.backward()
print(f"\nRMSNorm gradient norm: {x_test.grad.norm().item():.4f}")
print("RMSNorm provides well-scaled gradients regardless of input scale.")

## 4.6 Feed-Forward Networks with SwiGLU

Every transformer block contains a **position-wise feed-forward network (FFN)** applied independently to each token.

**Standard FFN** (original transformer, GPT-2):

$$\text{FFN}(\mathbf{x}) = \text{ReLU}(\mathbf{x}\mathbf{W}_1 + \mathbf{b}_1)\, \mathbf{W}_2 + \mathbf{b}_2$$

with $\mathbf{W}_1 \in \mathbb{R}^{d \times d_{ff}}$, $\mathbf{W}_2 \in \mathbb{R}^{d_{ff} \times d}$, $d_{ff} = 4d$.

**SwiGLU** (Shazeer, 2020, used in LLaMA, PaLM, GPT-4):

$$\text{SwiGLU}(\mathbf{x}) = (\text{SiLU}(\mathbf{x}\mathbf{W}_1) \odot \mathbf{x}\mathbf{W}_2)\, \mathbf{W}_3$$

where $\text{SiLU}(x) = x \cdot \sigma(x) = x / (1 + e^{-x})$ (Sigmoid-weighted Linear Unit).

The $\odot$ is element-wise multiplication — one branch acts as a **gate** that modulates the other.

**SwiGLU details:**
- 3 weight matrices instead of 2, but hidden dim is typically $2d_{ff}/3 \approx 8d/3$ to keep param count similar
- SiLU is smooth (non-zero gradient everywhere) unlike ReLU (dead neurons)
- Gating mechanism allows the network to zero out less relevant features selectively
- Empirically: +1-2 perplexity points better than ReLU-FFN at the same parameter budget

In [ ]:
# Standard FFN and SwiGLU FFN

torch.manual_seed(42)

class FFN(nn.Module):
    """Standard ReLU Feed-Forward Network."""
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.W1 = nn.Linear(d_model, d_ff)
        self.W2 = nn.Linear(d_ff, d_model)
    
    def forward(self, x):
        return self.W2(F.relu(self.W1(x)))

class SwiGLUFFN(nn.Module):
    """SwiGLU Feed-Forward Network (LLaMA-style)."""
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        # Use 2/3 * 4d to keep total params similar to standard FFN
        d_ff = d_ff or int(8 * d_model / 3)
        # Round to multiple of 256 for efficiency (common in practice)
        d_ff = ((d_ff + 255) // 256) * 256
        self.W1 = nn.Linear(d_model, d_ff, bias=False)  # gate (SiLU branch)
        self.W2 = nn.Linear(d_ff, d_model, bias=False)  # output projection
        self.W3 = nn.Linear(d_model, d_ff, bias=False)  # value branch
    
    def forward(self, x):
        # SiLU(xW1) * (xW3) then project
        gate = F.silu(self.W1(x))    # (B, T, d_ff) -- gating signal
        value = self.W3(x)           # (B, T, d_ff) -- value
        return self.W2(gate * value) # (B, T, d_model)

# Compare
B, T, d_model = 2, 10, 64
x = torch.randn(B, T, d_model)

ffn = FFN(d_model)
swiglu = SwiGLUFFN(d_model)

out_ffn = ffn(x)
out_swiglu = swiglu(x)

print(f"Input shape:        {x.shape}")
print(f"FFN output:         {out_ffn.shape}")
print(f"SwiGLU FFN output:  {out_swiglu.shape}")

# Parameter counts
ffn_params = sum(p.numel() for p in ffn.parameters())
swiglu_params = sum(p.numel() for p in swiglu.parameters())
print(f"\nFFN params:        {ffn_params:,}  (expected ~8*d^2 + bias = {8*d_model**2:,})")
print(f"SwiGLU FFN params: {swiglu_params:,}")

# SiLU vs ReLU comparison
print("\n--- SiLU vs ReLU activation comparison ---")
vals = torch.linspace(-3, 3, 7)
print(f"Input:  {vals.round(decimals=2).tolist()}")
print(f"ReLU:   {F.relu(vals).round(decimals=3).tolist()}")
print(f"SiLU:   {F.silu(vals).round(decimals=3).tolist()}")
print("(SiLU is smooth: non-zero for negative inputs, no 'dead neuron' problem)")

# Output distribution comparison
print(f"\nFFN output stats:    mean={out_ffn.mean():.4f}, std={out_ffn.std():.4f}")
print(f"SwiGLU output stats: mean={out_swiglu.mean():.4f}, std={out_swiglu.std():.4f}")

## 4.7 Full Transformer Block

A **transformer block** (decoder-only, as in GPT/LLaMA) combines attention and FFN with **pre-norm** and **residual connections**:

$$\mathbf{H} = \mathbf{x} + \text{MHA}(\text{RMSNorm}(\mathbf{x}))$$

$$\mathbf{H}' = \mathbf{H} + \text{FFN}(\text{RMSNorm}(\mathbf{H}))$$

**Residual connections** ($+\mathbf{x}$):
- Provide gradient highways (gradients flow directly from output to input)
- Allow the model to learn *corrections* rather than full transformations
- Critical for training deep networks (without residuals, gradients vanish)

**Pre-norm** (normalize before sub-layers):
- More stable training than original post-norm
- Input to each sub-layer has bounded scale
- Allows higher learning rates

**Full decoder stack:** $L$ transformer blocks stacked sequentially, each with the same structure but independent parameters. Final output passed through a projection head $\mathbf{W}_{\text{out}} \in \mathbb{R}^{d \times |V|}$ to produce logits over the vocabulary.

The complete transformer decoder forward pass:
$$\mathbf{H}^{(0)} = \text{Embed}(x) + \text{PE} \quad \longrightarrow \quad \mathbf{H}^{(l)} = \text{Block}_l(\mathbf{H}^{(l-1)}) \quad \longrightarrow \quad \text{logits} = \text{RMSNorm}(\mathbf{H}^{(L)})\mathbf{W}_{\text{out}}$$

In [ ]:
# Full Transformer Block with pre-norm and residual connections

torch.manual_seed(42)

class TransformerBlock(nn.Module):
    """Single transformer decoder block with pre-norm."""
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLUFFN(d_model, d_ff)
    
    def forward(self, x, mask=None):
        # Pre-norm attention with residual
        h, _ = self.attn(self.norm1(x), mask=mask)
        x = x + h                        # residual connection 1
        # Pre-norm FFN with residual
        x = x + self.ffn(self.norm2(x))  # residual connection 2
        return x

class TinyGPT(nn.Module):
    """Minimal GPT-style decoder-only transformer."""
    def __init__(self, vocab_size, d_model, num_heads, num_layers, max_seq_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        # Sinusoidal PE (no parameters)
        self.register_buffer('pe', sinusoidal_pe(max_seq_len, d_model))
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads)
            for _ in range(num_layers)
        ])
        self.norm_final = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
    
    def forward(self, x):
        B, T = x.shape
        # Causal mask
        causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)
        # Embeddings + PE
        h = self.embedding(x) + self.pe[:T].unsqueeze(0)  # (B, T, d)
        # Transformer blocks
        for block in self.blocks:
            h = block(h, mask=causal_mask)
        # Final norm and projection
        logits = self.lm_head(self.norm_final(h))  # (B, T, V)
        return logits

# Build and test
d_model = 64
num_heads = 4
num_layers = 4
vocab_size = 256
max_seq_len = 128
B, T = 2, 16

model = TinyGPT(vocab_size, d_model, num_heads, num_layers, max_seq_len)

token_ids = torch.randint(0, vocab_size, (B, T))
with torch.no_grad():
    logits = model(token_ids)

print(f"Input shape:  {token_ids.shape}")
print(f"Output shape: {logits.shape}  (B={B}, T={T}, V={vocab_size})")
print(f"\nLogits stats: mean={logits.mean():.4f}, std={logits.std():.4f}")

# Verify forward pass probabilities
probs = F.softmax(logits[0, 0], dim=-1)
print(f"Probs at pos 0: sum={probs.sum():.6f}, max={probs.max():.4f}, min={probs.min():.4f}")

# Stack depth
print(f"\nNumber of transformer blocks: {len(model.blocks)}")
print(f"Each block: MHA + SwiGLU FFN + 2x RMSNorm")

## 4.8 Parameter Count Verification

Understanding the parameter budget is essential for estimating training compute and memory.

**Per transformer layer (without bias):**

| Component | Formula | $d=1024, h=16$ |
|-----------|---------|----------------|
| MHA ($W^Q, W^K, W^V, W^O$) | $4d^2$ | 4,194,304 |
| FFN (SwiGLU: $W_1, W_2, W_3$) | $\approx 8d^2$ | 8,388,608 |
| LayerNorm / RMSNorm (×2) | $4d$ | 4,096 |
| **Total per layer** | **$\approx 12d^2$** | **12,582,912** |

**Full model:**

$$N \approx 12 L d^2 + V d$$

where the $Vd$ term is the embedding table (shared with lm_head in weight tying).

**Examples:**
- GPT-2 small: $L=12, d=768, V=50257 \Rightarrow \approx 117M$ params
- LLaMA-7B: $L=32, d=4096, V=32000 \Rightarrow \approx 7B$ params
- GPT-4 (estimated): $L\approx 120, d\approx 12288 \Rightarrow \approx 1.8T$ params

**Memory:** Each parameter in fp16/bf16 takes 2 bytes. A 7B model needs $\approx 14$ GB just for weights (plus activations, optimizer states, and gradients during training).

In [ ]:
# Parameter count verification

torch.manual_seed(42)

def count_params(model):
    """Count total and per-component parameters."""
    return sum(p.numel() for p in model.parameters())

def count_params_by_name(model):
    """Group parameters by component type."""
    groups = {}
    for name, p in model.named_parameters():
        # Classify by top-level component
        if 'embedding' in name or 'lm_head' in name:
            key = 'embedding+lm_head'
        elif 'attn' in name:
            key = 'attention'
        elif 'ffn' in name:
            key = 'ffn'
        elif 'norm' in name:
            key = 'norms'
        else:
            key = 'other'
        groups[key] = groups.get(key, 0) + p.numel()
    return groups

# Small model
L, d, V_sz = 4, 256, 1000
num_heads = 8

small_model = TinyGPT(
    vocab_size=V_sz,
    d_model=d,
    num_heads=num_heads,
    num_layers=L,
    max_seq_len=512
)

total = count_params(small_model)
groups = count_params_by_name(small_model)

print(f"Model config: L={L}, d={d}, h={num_heads}, V={V_sz}")
print(f"\nTotal parameters: {total:,}")
print("\nBreakdown:")
for k, v in sorted(groups.items(), key=lambda x: -x[1]):
    pct = 100 * v / total
    print(f"  {k:<20}: {v:>10,}  ({pct:.1f}%)")

# Formula verification
# Per layer: 4d^2 (MHA) + ~8d^2 (SwiGLU FFN) + 4d (2x RMSNorm) = 12d^2 + 4d
ffn_test = SwiGLUFFN(d)
ffn_params = count_params(ffn_test)
mha_test = MultiHeadAttention(d, num_heads)
mha_params = count_params(mha_test)

print(f"\nPer-component parameter counts (d={d}):")
print(f"  MHA:          {mha_params:,}  (4d^2 = {4*d**2:,})")
print(f"  SwiGLU FFN:   {ffn_params:,}  (~8d^2 = {8*d**2:,})")
print(f"  Per layer:    {mha_params + ffn_params + 2*d:,}  (~12d^2 = {12*d**2:,})")

# Scaling to real models
print("\n--- Parameter estimates for real models ---")
configs = [
    ("GPT-2 small",   12,  768,  50257),
    ("GPT-2 large",   36, 1280,  50257),
    ("LLaMA-7B",      32, 4096,  32000),
    ("LLaMA-13B",     40, 5120,  32000),
    ("LLaMA-70B",     80, 8192,  32000),
]
print(f"  {'Model':<16} {'L':>4} {'d':>6} {'V':>8} {'Est. Params':>14}")
print("  " + "-" * 52)
for name, l, d_m, v in configs:
    est = 12 * l * d_m**2 + v * d_m
    print(f"  {name:<16} {l:>4} {d_m:>6} {v:>8} {est/1e6:>10.0f}M  ({est/1e9:.1f}B)")

## 4.9 KV Cache Demo

**The problem:** Autoregressive generation is inherently sequential — to generate token $t+1$, we need the output from tokens $1, \ldots, t$. A naive implementation re-computes K and V for all past tokens at every step, giving $O(T^2 d)$ time complexity.

**KV Cache:** Cache the key and value tensors from past tokens and reuse them, computing only the new token's Q, K, V at each step.

At decode step $t$:
1. Compute $\mathbf{k}_t = \mathbf{x}_t \mathbf{W}^K$ and $\mathbf{v}_t = \mathbf{x}_t \mathbf{W}^V$ for the new token
2. **Append** to the cache: $\mathbf{K}_{\text{cache}} = [\mathbf{K}_{1:t-1}; \mathbf{k}_t]$
3. Attend: $\text{output}_t = \text{Attention}(\mathbf{q}_t, \mathbf{K}_{\text{cache}}, \mathbf{V}_{\text{cache}})$

**Memory cost:** $2 \cdot L \cdot T_{\max} \cdot d$ values (2 for K and V, per layer, per position, per dimension).

For LLaMA-7B generating 2048 tokens: $2 \times 32 \times 2048 \times 4096 \times 2$ bytes $\approx 1.07$ GB in fp16.

**Speedup:** Without KV cache, each token requires $O(Td)$ matrix multiplications. With cache, only $O(d)$ per new token. This gives $O(T)$ speedup for long sequences — critical for practical deployment.

**Techniques to reduce KV cache memory:**
- **Multi-Query Attention (MQA):** Single K,V shared across all heads
- **Grouped-Query Attention (GQA):** K,V shared within groups of heads (LLaMA-2/3)
- **Quantization:** Store K,V in int8/int4

In [ ]:
# KV Cache demonstration

torch.manual_seed(42)

class KVCache:
    """Simple KV cache for autoregressive decoding."""
    def __init__(self, num_layers, num_heads, d_k):
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.d_k = d_k
        # Cache: {layer_id: {'k': tensor, 'v': tensor}}
        self.cache = {layer: {'k': None, 'v': None} for layer in range(num_layers)}
    
    def update(self, layer_id, new_k, new_v):
        """
        Append new K, V to the cache for a given layer.
        new_k, new_v: (B, 1, num_heads, d_k) — single new token
        Returns full cached K, V: (B, T_cached, num_heads, d_k)
        """
        if self.cache[layer_id]['k'] is None:
            self.cache[layer_id]['k'] = new_k
            self.cache[layer_id]['v'] = new_v
        else:
            self.cache[layer_id]['k'] = torch.cat([self.cache[layer_id]['k'], new_k], dim=1)
            self.cache[layer_id]['v'] = torch.cat([self.cache[layer_id]['v'], new_v], dim=1)
        return self.cache[layer_id]['k'], self.cache[layer_id]['v']
    
    def get_seq_len(self):
        """Return current cached sequence length."""
        if self.cache[0]['k'] is None:
            return 0
        return self.cache[0]['k'].shape[1]
    
    def memory_bytes(self, dtype_bytes=2):
        """Estimate memory usage in bytes (assuming bf16/fp16)."""
        total = 0
        for layer_id in range(self.num_layers):
            for tensor in self.cache[layer_id].values():
                if tensor is not None:
                    total += tensor.numel() * dtype_bytes
        return total

# Demo: simulate decode steps with KV cache
B = 1
num_layers = 4
num_heads = 4
d_k = 16  # per-head dim
d_model = num_heads * d_k  # = 64

# Projection matrices
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)
W_q = nn.Linear(d_model, d_model, bias=False)

kv_cache = KVCache(num_layers, num_heads, d_k)

print("Simulating autoregressive decoding with KV cache...")
print(f"Config: {num_layers} layers, {num_heads} heads, d_k={d_k}")
print()

# Generate 8 tokens one at a time
for step in range(8):
    # New token embedding (1 token)
    x_new = torch.randn(B, 1, d_model)  # (B, 1, d_model)
    
    # Process through each layer
    for layer_id in range(num_layers):
        # Compute Q for new token: (B, 1, d_model)
        q_new = W_q(x_new).view(B, 1, num_heads, d_k)
        # Compute new K, V: (B, 1, num_heads, d_k)
        k_new = W_k(x_new).view(B, 1, num_heads, d_k)
        v_new = W_v(x_new).view(B, 1, num_heads, d_k)
        
        # Append to cache and get full K, V
        k_full, v_full = kv_cache.update(layer_id, k_new, v_new)
        
        # Attention: q_new attends to all cached K, V
        # q_new: (B, 1, h, d_k) -> (B*h, 1, d_k)
        # k_full: (B, T, h, d_k) -> (B*h, T, d_k)
        T_cached = k_full.shape[1]
        q_attn = q_new.transpose(1, 2).reshape(B * num_heads, 1, d_k)
        k_attn = k_full.transpose(1, 2).reshape(B * num_heads, T_cached, d_k)
        v_attn = v_full.transpose(1, 2).reshape(B * num_heads, T_cached, d_k)
        
        scores = torch.bmm(q_attn, k_attn.transpose(1, 2)) / math.sqrt(d_k)
        attn_weights = F.softmax(scores, dim=-1)  # (B*h, 1, T_cached)
        context = torch.bmm(attn_weights, v_attn)  # (B*h, 1, d_k)
    
    mem_kb = kv_cache.memory_bytes() / 1024
    print(f"  Step {step+1}: cached_len={kv_cache.get_seq_len()}, "
          f"KV cache memory={mem_kb:.2f} KB, "
          f"attn_weights shape={attn_weights.shape}")

# Memory scaling analysis
print("\n--- KV Cache memory scaling (fp16, 1 sequence) ---")
print(f"  {'seq_len':>10} {'L=12 d=768':>14} {'L=32 d=4096':>14}")
print("  " + "-" * 42)
for T_len in [128, 512, 2048, 8192, 32768]:
    # Memory = 2 (K+V) * L * T * d * 2 bytes
    mem_small = 2 * 12 * T_len * 768 * 2  # GPT-2 large
    mem_large = 2 * 32 * T_len * 4096 * 2  # LLaMA-7B
    print(f"  {T_len:>10,} {mem_small/1e6:>10.1f} MB {mem_large/1e6:>10.1f} MB")

## Chapter 4 Summary

| Component | Key Formula | Purpose |
|-----------|------------|----------|
| Token Embedding | $\mathbf{x}_t = \mathbf{E}[id_t]$ | Map tokens to vectors |
| Sinusoidal PE | $\sin(pos/10000^{2i/d})$ | Encode position |
| Scaled Attention | $\text{softmax}(QK^T/\sqrt{d_k})V$ | Context aggregation |
| Causal Mask | $-\infty$ upper triangle | Autoregressive property |
| Multi-Head | $\text{Concat}(\text{head}_1,\ldots,\text{head}_h)W^O$ | Multi-aspect attention |
| RoPE | Rotate $(q, k)$ pairs by position angle | Relative position encoding |
| RMSNorm | $x / \text{RMS}(x) \cdot \gamma$ | Activation normalization |
| SwiGLU | $\text{SiLU}(xW_1) \odot xW_2 \cdot W_3$ | Gated FFN |
| Residual | $x + \text{sublayer}(x)$ | Gradient flow |
| KV Cache | Reuse past K, V tensors | Efficient decoding |

**Key takeaways:**
- Attention is the core mechanism: $O(T^2 d)$ complexity makes long contexts expensive
- Scaling by $\sqrt{d_k}$ prevents softmax saturation — numerically critical
- Each transformer layer has $\approx 12d^2$ parameters; full model $\approx 12Ld^2 + Vd$
- RoPE elegantly encodes relative positions through complex-valued rotations
- KV cache trades memory for compute — essential for production inference

**Next:** Chapter 5 covers optimization algorithms — AdamW, gradient clipping, learning rate schedules, and the Chinchilla scaling laws.